In [0]:
from pyspark.sql.functions import current_timestamp,rand, expr

base_path = "/Volumes/workspace/default/lakehouse"
landing_path = f"{base_path}/landing/sales"
bronze_path = f"{base_path}/bronze/sales"
silver_path = f"{base_path}/silver/sales"
gold_path = f"{base_path}/gold/sales"

spark.range(1000).withColumn("transaction_id", expr("uuid()")) \
    .withColumn("user_id", (rand() * 100).cast("int")) \
    .withColumn("amount", (rand() * 500)-50) \
    .withColumn("event_time",current_timestamp()) \
    .write.format("json").mode("overwrite").save(landing_path)

## Bronze layer

In [0]:
from pyspark.sql.functions import col, current_timestamp

raw_df = spark.read.format("json").load(landing_path)

bronze_df = raw_df.withColumn("ingest_time", current_timestamp()) \
                    .withColumn("source_file", col("_metadata.file_path"))

bronze_df.write.format("delta").mode("append").saveAsTable("workspace.default.sales_bronze")

bronze_df.display()


## Silver layer

In [0]:
bronze_stream = spark.read.format("delta").load(bronze_path)

silver_clean_df = bronze_stream \
    .filter("user_id is not null and amount > 0") \
        .dropDuplicates(["transaction_id"])

silver_clean_df.write.format("delta").mode("overwrite").save(silver_path)

silver_clean_df.write.format("delta").mode("append").saveAsTable("workspace.default.sales_silver")

silver_clean_df.display()


## Gold layer

In [0]:
from pyspark.sql.functions import sum,count,round

silver_df = spark.read.format("delta").load(silver_path)

gold_agg_df = silver_clean_df.groupBy("user_id") \
    .agg(round(sum("amount"),2).alias("total_spent"),
         count("transaction_id").alias("transaction_count")
         )

gold_agg_df.write.format("delta").mode("overwrite").save(gold_path)

gold_agg_df.write.format("delta").mode("append").saveAsTable("workspace.default.sales_gold") 

gold_agg_df.display()